# 01 — Data Ingestion & Delta Lake
**SF Crime Classification | Databricks Pipeline — Step 1 of 4**

### Before running:
1. **Catalog** (left sidebar) → expand **workspace** → **default** → **Volumes**
2. Click the **sf_crime_data** volume → **Upload to this volume** → upload `train.csv` and `test.csv`
3. Run this notebook top to bottom.


In [ ]:
CATALOG = "workspace"
SCHEMA  = "default"
VOLUME  = "sf_crime_data"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
TRAIN_PATH  = f"{VOLUME_PATH}/train.csv"
TEST_PATH   = f"{VOLUME_PATH}/test.csv"

print(f"Volume : {VOLUME_PATH}")
print(f"Train  : {TRAIN_PATH}")
print(f"Test   : {TEST_PATH}")


In [ ]:
import os
for p in [TRAIN_PATH, TEST_PATH]:
    if os.path.exists(p):
        print(f"✅ Found: {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
    else:
        print(f"❌ Missing: {p} — upload via Catalog > workspace > default > Volumes > sf_crime_data")


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

train_raw = spark.read.option("header", True).option("inferSchema", True).csv(TRAIN_PATH)
test_raw  = spark.read.option("header", True).option("inferSchema", True).csv(TEST_PATH)

train_raw = train_raw.withColumn("Dates", F.col("Dates").cast(TimestampType()))
test_raw  = test_raw.withColumn("Dates",  F.col("Dates").cast(TimestampType()))

print(f"Train: {train_raw.count():,} rows, {len(train_raw.columns)} cols")
print(f"Test : {test_raw.count():,} rows,  {len(test_raw.columns)} cols")
train_raw.printSchema()


In [ ]:
for c in ["Category", "PdDistrict", "DayOfWeek", "Address"]:
    train_raw = train_raw.withColumn(c, F.upper(F.trim(F.col(c))))
for c in ["PdDistrict", "DayOfWeek", "Address"]:
    test_raw = test_raw.withColumn(c, F.upper(F.trim(F.col(c))))
print("✅ Text columns standardised")


In [ ]:
before = train_raw.count()
train_clean = train_raw.filter(
    F.col("X").between(-122.55, -122.33) & F.col("Y").between(37.65, 37.85)
)
print(f"Removed {before - train_clean.count():,} geographic outliers")

before = train_clean.count()
train_clean = train_clean.dropDuplicates()
print(f"Removed {before - train_clean.count():,} duplicates")
print(f"Final train: {train_clean.count():,} rows")

test_clean = test_raw


In [ ]:
(train_clean.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.train_raw"))

(test_clean.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.test_raw"))

print(f"✅ {CATALOG}.{SCHEMA}.train_raw")
print(f"✅ {CATALOG}.{SCHEMA}.test_raw")


In [ ]:
df = spark.read.table(f"{CATALOG}.{SCHEMA}.train_raw")
print(f"Train Delta rows: {df.count():,}")
df.groupBy("Category").count().orderBy(F.desc("count")).show(10, truncate=False)


---
**Next:** Run `02_feature_engineering.ipynb`.